# Εκπαίδευση SVM μοντέλων

Σύμφωνα με την εκφώνηση εκπαιδεύουμε μοντέλα Μηχανικής Μάθησης Random Forest και SVM.

Στο notebook αυτό εκπαιδεύουμε SVM μοντέλα για κάθε dataset, ώστε να συγκριθούν με τα αντίστοιχα Random Forest.

In [1]:
import sys, joblib
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix

from src.features import FEATURE_NAMES

MODELS = Path('../models')

def make_svm():
    return Pipeline([
        ('scaler', RobustScaler()),
        ('clf', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True,
                    class_weight='balanced', random_state=42)),
    ])

def evaluate(pipe, X, y, groups, name, n_splits=5):
    cv = GroupKFold(min(n_splits, len(np.unique(groups))))
    y_proba = cross_val_predict(pipe, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    print(f'{name}: acc={accuracy_score(y, y_pred):.3f}, F1={f1_score(y, y_pred):.3f}, MCC={matthews_corrcoef(y, y_pred):.3f}')
    cm = confusion_matrix(y, y_pred)
    print(f'  CM: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

## SVM για UCI balanced

In [2]:
uci = pd.read_csv('../data/uci/pd_speech_features.csv', header=1)
rng = np.random.default_rng(42)
hc_subj = uci[uci['class']==0]['id'].unique()
pd_subj = rng.choice(uci[uci['class']==1]['id'].unique(), size=len(hc_subj), replace=False)
uci_bal = uci[uci['id'].isin(np.concatenate([hc_subj, pd_subj]))].reset_index(drop=True)

X = uci_bal[FEATURE_NAMES]
y = uci_bal['class'].values
groups = uci_bal['id'].values

pipe = make_svm()
evaluate(pipe, X, y, groups, 'UCI-balanced SVM', n_splits=10)
pipe.fit(X, y)
joblib.dump(pipe, MODELS / 'uci_balanced_svm.joblib')

UCI-balanced SVM: acc=0.659, F1=0.656, MCC=0.318
  CM: TN=128, FP=64, FN=67, TP=125


['../models/uci_balanced_svm.joblib']

## SVM για Iyer (8 kHz)

In [3]:
iyer = pd.read_csv('../data/iyer/iyer_features_8khz.csv')
X = iyer[FEATURE_NAMES]
y = iyer['class'].values
groups = iyer['subject'].values

pipe = make_svm()
evaluate(pipe, X, y, groups, 'Iyer 8kHz SVM')
pipe.fit(X, y)
joblib.dump(pipe, MODELS / 'iyer_8khz_svm.joblib')

Iyer 8kHz SVM: acc=0.642, F1=0.651, MCC=0.285
  CM: TN=25, FP=16, FN=13, TP=27


['../models/iyer_8khz_svm.joblib']

## SVM για MDVR

In [4]:
mdvr = pd.read_csv('../data/mdvr_kcl/mdvr_features.csv')
X = mdvr[FEATURE_NAMES]
y = mdvr['class'].values
groups = mdvr['subject'].values

pipe = make_svm()
evaluate(pipe, X, y, groups, 'MDVR SVM')
pipe.fit(X, y)
joblib.dump(pipe, MODELS / 'mdvr_svm.joblib')

print('\nAll SVM models saved.')

MDVR SVM: acc=0.699, F1=0.645, MCC=0.383
  CM: TN=31, FP=11, FN=11, TP=20

All SVM models saved.
